# 📨 Telegram Tools — Quick Start (Colab)

Run any of the four tools (copy, forward, extract, process) directly from Colab.

## Instructions
1. Run Cell 1 (install)
2. Edit credentials in Cell 2
3. Run Cell 3 (login — saves SESSION_STRING)
4. Run any of Cells 4-7 for the operation you need

## ⚠️ Security
- Clear outputs before saving/sharing the notebook
- Don't commit credentials to Git
- Revoke the API app at my.telegram.org when done

In [ ]:
# Cell 1: Install telegram-tools
!pip install -q telethon gradio cryptg hachoir
!pip install -q git+https://github.com/DrAbdulmalek/telegram-tools.git
print('Done! Import telegram_tools to verify:')
import telegram_tools
print(f'Version: {telegram_tools.__version__}')

In [ ]:
# Cell 2: Your credentials — EDIT THESE

API_ID = 12345678                      # from my.telegram.org
API_HASH = 'your_api_hash_here'         # from my.telegram.org
PHONE = '+963XXXXXXXXX'                # with country code

import os
os.environ['TG_API_ID'] = str(API_ID)
os.environ['TG_API_HASH'] = API_HASH
os.environ['TG_PHONE'] = PHONE

assert isinstance(API_ID, int) and API_ID > 0, 'API_ID must be a positive integer'
assert isinstance(API_HASH, str) and len(API_HASH) >= 10, 'Invalid API_HASH'
print(f'Ready: API_ID={API_ID}, PHONE={PHONE}')

In [ ]:
# Cell 3: Login and export SESSION_STRING
# Run this once, save the output string, then skip this cell in future runs

import asyncio
from telegram_tools.core.forwarder import TelegramForwarder

async def login():
    fwd = TelegramForwarder(API_ID, API_HASH)
    await fwd._ensure_client()
    if await fwd.is_authorized():
        print('Already authenticated')
    else:
        await fwd.send_code(PHONE)
        code = input('Enter code from Telegram: ').strip()
        try:
            await fwd.verify_code(code)
        except Exception as e:
            if '2FA_PASSWORD_REQUIRED' in str(e):
                password = input('Enter 2FA password: ').strip()
                await fwd.verify_code(code, password)
            else:
                raise
    ss = await fwd.export_session_string()
    print('\n=== SESSION_STRING ===')
    print(ss)
    print('=== END ===')
    await fwd.disconnect()

await login()

In [ ]:
# Cell 4: Copy (fast bulk — for public channels)

SOURCE = '@public_channel'
DEST = -1001234567890      # your private channel ID
LIMIT = 100               # 0 = all
DELAY = 3                 # seconds

from telegram_tools.core.copier import TelegramCopier, CopierConfig

async def run_copy():
    copier = TelegramCopier(API_ID, API_HASH)
    await copier._ensure_client()
    if not await copier.is_authorized():
        await copier.send_code(PHONE)
        code = input('Code: ').strip()
        await copier.verify_code(code)
    config = CopierConfig(source_channel=SOURCE, dest_channel=str(DEST),
                          limit=LIMIT, delay=DELAY)
    result = await copier.copy(config)
    print(result.to_dict())
    await copier.disconnect()

await run_copy()

In [ ]:
# Cell 5: Forward (bypass 'Restrict Saving' — for protected channels)

SOURCE = '@protected_channel'
DEST = -1001234567890
LIMIT = 50
DELAY = 3

from telegram_tools.core.forwarder import TelegramForwarder, ForwardConfig

async def run_forward():
    fwd = TelegramForwarder(API_ID, API_HASH)
    await fwd._ensure_client()
    if not await fwd.is_authorized():
        await fwd.send_code(PHONE)
        code = input('Code: ').strip()
        await fwd.verify_code(code)
    config = ForwardConfig(source_channel=SOURCE, dest_channel=str(DEST),
                           limit=LIMIT, delay=DELAY)
    result = await fwd.forward_content(config)
    print(result.to_dict())
    await fwd.disconnect()

await run_forward()

In [ ]:
# Cell 6: Extract corpus (texts + media)

CHANNEL = '@my_channel'
OUTPUT = './telegram_corpus'
LIMIT = 500
TEXTS_ONLY = False  # True = skip media download

from telegram_tools.core.extractor import TelegramExtractor

async def run_extract():
    ex = TelegramExtractor(API_ID, API_HASH)
    await ex._ensure_client()
    if not await ex.is_authorized():
        await ex.send_code(PHONE)
        code = input('Code: ').strip()
        await ex.verify_code(code)
    metadata = await ex.extract(
        channel=CHANNEL, output_dir=OUTPUT,
        texts_only=TEXTS_ONLY, limit=LIMIT,
    )
    print(metadata)
    await ex.disconnect()

await run_extract()

In [ ]:
# Cell 7: Process extracted corpus (Arabic normalization + dedup + segment)

from telegram_tools.core.preprocess import CorpusProcessor

processor = CorpusProcessor('./telegram_corpus', './processed_corpus')
stats = processor.process()
print(stats)

## Notes

- Keep this page open during execution
- For large channels (1000+ messages), use `DELAY >= 5`
- Files over 2GB may timeout — use `LIMIT` to batch
- Clear outputs (Runtime → Clear output) before saving/sharing